<a href="https://colab.research.google.com/github/DariaDubovska/quora-duplicate-question-detection/blob/main/Quora_duplicates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Libraries importing

In [ ]:
!pip install -q spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 72.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import pandas as pd
import numpy as np
import re
import html
import spacy
from collections import Counter
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                            recall_score, roc_auc_score, classification_report,
                            confusion_matrix)
from transformers import BertTokenizer, BertModel
import torch
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

import os
import pickle

In [ ]:
RANDOMSEED = 42

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/ML Projects/Quora duplicates"
cache_file = os.path.join(PROJECT_DIR, "embeddings/bert_embeddings.pkl")
chunks_dir = os.path.join(PROJECT_DIR, "embeddings/bert_cache_chunks")
feature_file = os.path.join(PROJECT_DIR, "data/processed/quora_with_bert_cosine_similarity.csv")
tfidf_save_path = os.path.join(PROJECT_DIR, "embeddings/tfidf_text_map.pkl")

## Data Loading

In [ ]:
df = pd.read_csv("quora_question_pairs_train.csv.zip", index_col=0)
pd.set_option("display.max_colwidth", None)
df.head()

,qid1,qid2,question1,question2,is_duplicate
id,,,,,
332278,459256,459257,The Iliad and the Odyssey in the Greek culture?,How do I prove that the pairs of three independent variables is also independent?,0
196656,297402,297403,What is practical management and what is strategic management?,What are the practical aspects of strategic management?,0
113125,184949,184950,How useful is MakeUseOf Answers?,"Is there any Q&A site that is not Yahoo answers, where hate speech is allowed?",0
266232,101283,163744,Which is the best place to reside in India and Why?,Which ia the best place to visit in India?,0
122738,17811,27517,Why do so many people ask questions on Quora that can be easily answered by any number of legitimate sources on the Web? Have they not heard of Google or Bing?,Why don't many people posting questions on Quora check Google first?,1


In [ ]:
df_test = pd.read_csv("quora_question_pairs_test.csv.zip", index_col=0)
df_test.head()

,qid1,qid2,question1,question2,is_duplicate
id,,,,,
305985,429434,429435,Why is beef banned in India and not pork as well?,Is beef banned in india?,0
5193,10230,10231,At what valuation did Homejoy raise money in December of 2013?,"Should a wealthy founder self-fund his second startup then raise money at high valuation after getting traction, or raise money at low valuation before any traction?",0
123326,199422,199423,How do we judge?,How do I judge my love?,0
368557,327674,498931,Are Adderall and meth the same?,Are concerta and meth test the same?,0
369226,499645,499646,"If you had internet access to only one site for the rest of your life, which site would you pick?",Why is there .co.uk for British internet sites but only .fr for French ones?,0


# Explonatory Data Analysis

In [ ]:
df.shape

(323432, 5)

In [ ]:
df.isnull().sum()

,0
qid1,0
qid2,0
question1,1
question2,2
is_duplicate,0


In [ ]:
df[df["question1"].isna()]

,qid1,qid2,question1,question2,is_duplicate
id,,,,,
363362,493340,493341,NaN,My Chinese name is Haichao Yu. What English name is most suitable for me considering the pronounciation of my Chinese name?,0


In [ ]:
df[df["question2"].isna()]

,qid1,qid2,question1,question2,is_duplicate
id,,,,,
201841,303951,174364,How can I create an Android app?,NaN,0
105780,174363,174364,How can I develop android app?,NaN,0


**Dropping entries with missing values**

In [ ]:
df.dropna(subset=["question1", "question2"], inplace=True)
df.reset_index(drop=True, inplace=True)

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
((df["question1"].str.strip() == "") |
 (df["question2"].str.strip() == "")).sum()

np.int64(0)

**There is no duplicates or empty strings in dataset**

In [ ]:
df["is_duplicate"].value_counts(normalize=True)

,proportion
is_duplicate,
0,0.6308
1,0.3692


In [ ]:
print("Percentage Distribution of",(df["is_duplicate"].value_counts()/df["is_duplicate"].count())*100)

Percentage Distribution of is_duplicate
0    63.079996
1    36.920004
Name: count, dtype: float64


In [ ]:
questions = pd.concat([df["question1"], df["question2"]], ignore_index=True)
questions.str.len().describe()

,0
count,646858.000000
mean,59.827502
std,31.968420
min,1.000000
25%,39.000000
50%,51.000000
75%,72.000000
max,1169.000000


In [ ]:
questions.str.len().quantile([0.95, 0.99, 0.999])

,0
0.950,125.00
0.990,156.00
0.999,270.14


**Most questions are short: the mean length is around 60 characters, and 99% of questions are shorter than roughly 270 characters. This supports using `max_length=128` for BERT tokenization as a practical trade-off.**

In [ ]:
print("Number of unique questions:", questions.nunique())

Number of unique questions: 449361


In [ ]:
print("Number of all questions: ", questions.shape[0])

Number of all questions:  646858


## Data Split

In [ ]:
X = df.drop(columns=["is_duplicate"])
y = df['is_duplicate']

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, stratify = y,
                                                  random_state=RANDOMSEED)

In [ ]:
X_test = df_test.drop(columns=["is_duplicate"])
y_test = df_test['is_duplicate']

# TEXT PREPROCESSING + TF - IDF

## Text Cleaning

In [ ]:
def normalize_numbers(text: str):
    """Normalize abbreviated numeric forms while preserving their value."""

    # 1,000 → 1000
    text = re.sub(r"(?<=\d),(?=\d)", "", text)

    def replace_number(match):
        value = float(match.group(1))
        suffix = match.group(2).lower()

        multiplier = {
            "k": 1_000,
            "m": 1_000_000,
            "b": 1_000_000_000,
        }[suffix]

        normalized = value * multiplier

        if normalized.is_integer():
            return str(int(normalized))

        return str(normalized)

    text = re.sub(
        r"\b(\d+(?:\.\d+)?)\s*([kmb])\b",
        replace_number,
        text,
        flags=re.IGNORECASE
    )

    return text

In [ ]:
def prepare_text_for_tfidf(text):
    """
    Basic normalization before spaCy preprocessing for TF-IDF.
    """

    if pd.isna(text):
        return ""

    text = str(text)

    # HTML entities: &amp; → &
    text = html.unescape(text)

    # HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # URLs
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # 10k → 10000, 2.5m → 2500000
    text = normalize_numbers(text)

    # Line breaks and tabs
    text = re.sub(r"[\n\r\t]+", " ", text)

    # Multiple spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
def prepare_text_for_features(text):
    """
    Minimal text normalization for manual feature engineering.
    Stop words, punctuation and numbers are preserved.
    """

    if pd.isna(text):
        return ""

    text = str(text).lower()

    # &amp; → &
    text = html.unescape(text)

    # Remove HTML tags, but preserve the surrounding text
    text = re.sub(r"<[^>]+>", " ", text)

    # Normalize line breaks and tabs
    text = re.sub(r"[\n\r\t]+", " ", text)

    # Normalize multiple spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

# Lemmatization and Stop-Words

In [ ]:
nlp = spacy.load(
    "en_core_web_sm",
    disable=["parser", "ner"]
)

In [ ]:
NEGATION_WORDS = {
    "no",
    "not",
    "nor",
    "never",
    "neither"
}

for word in NEGATION_WORDS:
    nlp.vocab[word].is_stop = False

In [ ]:
def process_spacy_doc(doc):
    """
    Remove spaces, punctuation and stop words,
    preserve negations, and lemmatize the remaining tokens.
    """

    tokens = []

    for token in doc:
        if token.is_space or token.is_punct:
            continue

        lemma = token.lemma_.lower().strip()

        if lemma:
            tokens.append(lemma)

    return " ".join(tokens)

In [ ]:
for X in [X_train, X_valid, X_test]:
    X["question1_features"] = (
        X["question1"]
        .apply(prepare_text_for_features)
    )

    X["question2_features"] = (
        X["question2"]
        .apply(prepare_text_for_features)
    )

## TF-IDF text creating

In [ ]:
def collect_unique_questions(*dataframes):
    questions = []

    for df in dataframes:
        questions.append(df["question1"])
        questions.append(df["question2"])

    return (
        pd.concat(questions, ignore_index=True)
        .fillna("")
        .astype(str)
        .drop_duplicates()
        .reset_index(drop=True)
    )

In [ ]:
unique_questions = collect_unique_questions(
    X_train,
    X_valid,
    X_test
)

In [ ]:
basic_tfidf_texts = unique_questions.apply(
    prepare_text_for_tfidf
)

In [ ]:
processed_tfidf_texts = []

docs = nlp.pipe(
    basic_tfidf_texts,
    batch_size=1000,
    n_process=1
)

for doc in tqdm(
    docs,
    total=len(basic_tfidf_texts)
):
    processed_tfidf_texts.append(
        process_spacy_doc(doc)
    )

In [ ]:
tfidf_text_map = dict(
    zip(unique_questions, processed_tfidf_texts)
)

print(f"Processed questions: {len(tfidf_text_map):,}")

Processed questions: 537,359


In [ ]:
tfidf_save_path = tfidf_save_path.replace(
    ".pkl",
    "_v1.pkl"
)

In [ ]:
with open(tfidf_save_path, "wb") as f:
    pickle.dump(tfidf_text_map, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved to: {tfidf_save_path}")

Saved to: /content/drive/MyDrive/ML Projects/Quora duplicates/embeddings/tfidf_text_map_v1.pkl


## Load TF-IDF texts and add them as features to datasets

In [ ]:
with open(tfidf_save_path, "rb") as f:
    tfidf_text_map = pickle.load(f)

print(f"Loaded processed questions: {len(tfidf_text_map):,}")

Loaded processed questions: 537,359


In [ ]:
X_train["question1_tfidf"] = (
    X_train["question1"]
    .fillna("")
    .astype(str)
    .map(tfidf_text_map)
)

X_train["question2_tfidf"] = (
    X_train["question2"]
    .fillna("")
    .astype(str)
    .map(tfidf_text_map)
)

X_valid["question1_tfidf"] = (
    X_valid["question1"]
    .fillna("")
    .astype(str)
    .map(tfidf_text_map)
)

X_valid["question2_tfidf"] = (
    X_valid["question2"]
    .fillna("")
    .astype(str)
    .map(tfidf_text_map)
)

X_test["question1_tfidf"] = (
    X_test["question1"]
    .fillna("")
    .astype(str)
    .map(tfidf_text_map)
)

X_test["question2_tfidf"] = (
    X_test["question2"]
    .fillna("")
    .astype(str)
    .map(tfidf_text_map)
)

In [ ]:
text_columns = [
    "question1_tfidf",
    "question2_tfidf"
]

for column in text_columns:
    print(
        column,
        "train empty:",
        X_train[column].fillna("").str.strip().eq("").sum(),
        "val empty:",
        X_valid[column].fillna("").str.strip().eq("").sum()
    )

question1_tfidf train empty: 7 val empty: 4
question2_tfidf train empty: 4 val empty: 1


An initial preprocessing pipeline included stop-word removal. After inspecting the processed texts, it became clear that this step removed all informative tokens from some valid questions (e.g., "Who am I?", "How is everyone?", "Why did you give up?"), resulting in empty documents. Since interrogative words and auxiliary verbs carry important semantic information in question-pair similarity tasks, stop-word removal was excluded from the final pipeline. After this revision, only questions consisting entirely of punctuation or other non-lexical characters remained empty. These samples were retained, as TF-IDF naturally represents them as zero vectors without affecting the preprocessing pipeline.

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=False,
    analyzer="word",
    token_pattern=r"(?u)\b\w+\b",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    norm="l2",
    dtype=np.float32
)

In [ ]:
train_q1 = X_train["question1_tfidf"].fillna("")
train_q2 = X_train["question2_tfidf"].fillna("")

valid_q1 = X_valid["question1_tfidf"].fillna("")
valid_q2 = X_valid["question2_tfidf"].fillna("")

In [ ]:
train_corpus = pd.concat(
    [
        train_q1,
        train_q2
    ],
    ignore_index=True
)

tfidf_vectorizer.fit(train_corpus)

In [ ]:
print(
    f"Vocabulary size: {len(tfidf_vectorizer.vocabulary_):,}"
)

Vocabulary size: 342,949


In [ ]:
X_train_q1 = tfidf_vectorizer.transform(train_q1)
X_train_q2 = tfidf_vectorizer.transform(train_q2)

X_valid_q1 = tfidf_vectorizer.transform(valid_q1)
X_valid_q2 = tfidf_vectorizer.transform(valid_q2)

In [ ]:
X_train_diff = abs(X_train_q1 - X_train_q2)
X_valid_diff = abs(X_valid_q1 - X_valid_q2)

X_train_product = X_train_q1.multiply(X_train_q2)
X_valid_product = X_valid_q1.multiply(X_valid_q2)

X_train_pair = hstack(
    [X_train_diff, X_train_product],
    format="csr"
)

X_valid_pair = hstack(
    [X_valid_diff, X_valid_product],
    format="csr"
)

print("Train pair shape:", X_train_pair.shape)
print("Validation pair shape:", X_valid_pair.shape)

Train pair shape: (258743, 685898)
Validation pair shape: (64686, 685898)


In [ ]:
tfidf_lr = LogisticRegression(
    solver="saga",
    max_iter=300,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

In [ ]:
tfidf_lr.fit(
    X_train_pair,
    y_train
)

valid_pred = tfidf_lr.predict(X_valid_pair)
valid_proba = tfidf_lr.predict_proba(X_valid_pair)[:, 1]

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.


convergence after 24 epochs took 14 seconds


In [ ]:
print(
    classification_report(
        y_valid,
        valid_pred,
        digits=4
    )
)

print("Accuracy:", accuracy_score(y_valid, valid_pred))
print("F1:", f1_score(y_valid, valid_pred))
print("ROC-AUC:", roc_auc_score(y_valid, valid_proba))

print(
    "Confusion matrix:\n",
    confusion_matrix(y_valid, valid_pred)
)

              precision    recall  f1-score   support

           0     0.8372    0.8976    0.8663     40804
           1     0.8005    0.7017    0.7478     23882

    accuracy                         0.8253     64686
   macro avg     0.8188    0.7996    0.8071     64686
weighted avg     0.8236    0.8253    0.8226     64686

Accuracy: 0.8252790402869246
F1: 0.7478132809710818
ROC-AUC: 0.9039001528000858
Confusion matrix:
 [[36627  4177]
 [ 7125 16757]]


### Conclusions

- TF-IDF combined with Logistic Regression provides a strong supervised baseline for duplicate-question detection.
- The model achieved a validation ROC-AUC of approximately **0.90**, demonstrating a strong ability to distinguish duplicate and non-duplicate question pairs.
- Precision and recall are reasonably balanced, resulting in a positive-class F1-score of approximately **0.75**.
- The use of both unigram and bigram TF-IDF features allowed the model to capture both lexical overlap and short phrase similarity between questions.
- This model will serve as the baseline for comparison with more advanced approaches in the subsequent experiments.

# BERT EMBEDDING

This section uses pretrained `bert-base-uncased` embeddings as a feature extractor. The model is not fine-tuned here. Cosine similarity between question embeddings is used as a simple semantic baseline.

In [ ]:
!pip install transformers torch --quiet

In [ ]:
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model = bert_model.to(device)
bert_model.eval()

device

device(type='cuda')

## BERT embedding cache

This section loads already computed BERT embeddings from Google Drive.

In [ ]:
if os.path.exists(cache_file):
    with open(cache_file, "rb") as f:
        embedding_dict = pickle.load(f)
    print(f"Loaded {len(embedding_dict)} embeddings.")
else:
    embedding_dict = {}
    print("Creating new cache.")

Loaded 449361 embeddings.


### Compute missing embeddings only

Run this cell only if `Questions to process` is greater than 0. New embeddings are saved in chunk files, so progress is not lost if Colab disconnects.

In [ ]:
def get_sentence_embeddings_batch(texts, max_length=128):
    texts = [str(text) for text in texts]

    inputs = bert_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = bert_model(**inputs)

    last_hidden_states = outputs.last_hidden_state

    attention_mask = inputs["attention_mask"]
    mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_states.size()).float()

    embeddings = torch.sum(last_hidden_states * mask_expanded, dim=1) / torch.clamp(
        mask_expanded.sum(dim=1),
        min=1e-9
    )

    return embeddings.cpu().numpy()

In [ ]:
questions_to_process = [q for q in questions.unique() if q not in embedding_dict]
print(f"Questions to process: {len(questions_to_process)}")

In [ ]:
os.makedirs(chunks_dir, exist_ok=True)

batch_size = 64
chunk_size = 5000

chunk_id = len([f for f in os.listdir(chunks_dir) if f.endswith(".pkl")])
current_chunk = {}

for batch_start in tqdm(range(0, len(questions_to_process), batch_size)):
    batch_questions = questions_to_process[batch_start:batch_start + batch_size]

    batch_embeddings = get_sentence_embeddings_batch(batch_questions)

    for question, embedding in zip(batch_questions, batch_embeddings):
        embedding_dict[question] = embedding
        current_chunk[question] = embedding

    if len(current_chunk) >= chunk_size:
        chunk_path = os.path.join(chunks_dir, f"chunk_{chunk_id}.pkl")

        with open(chunk_path, "wb") as f:
            pickle.dump(current_chunk, f)

        print(f"Saved {chunk_path} with {len(current_chunk)} embeddings")

        current_chunk = {}
        chunk_id += 1

if len(current_chunk) > 0:
    chunk_path = os.path.join(chunks_dir, f"chunk_{chunk_id}.pkl")

    with open(chunk_path, "wb") as f:
        pickle.dump(current_chunk, f)

    print(f"Saved final {chunk_path} with {len(current_chunk)} embeddings")

In [ ]:
with open(cache_file, "wb") as f:
    pickle.dump(embedding_dict, f)

print(f"Main cache updated: {len(embedding_dict)} embeddings")

Main cache updated: 449361 embeddings


## BERT + Cosine similarity Experiment

In [ ]:
def cosine_similarity_embeddings(emb1, emb2):
     return cosine_similarity(
        emb1.reshape(1, -1),
        emb2.reshape(1, -1)
    )[0, 0]

In [ ]:
tqdm.pandas()

df["bert_cosine_similarity"] = df.progress_apply(
    lambda row: cosine_similarity_embeddings(
        embedding_dict[str(row["question1"])],
        embedding_dict[str(row["question2"])]
    ),
    axis=1
)

df[["question1", "question2", "is_duplicate", "bert_cosine_similarity"]].head()

In [41]:
train_scores = df.loc[X_train.index, "bert_cosine_similarity"]
valid_scores = df.loc[X_valid.index, "bert_cosine_similarity"]

best_threshold = None
best_f1 = -1

for threshold in np.arange(0.1, 1.0, 0.01):
    train_preds = (train_scores >= threshold).astype(int)
    f1 = f1_score(y_train, train_preds)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

valid_preds = (valid_scores >= best_threshold).astype(int)
valid_probs = (valid_scores + 1) / 2

print(f"Best threshold from train: {best_threshold:.2f}")
print(f"Train F1 at best threshold: {best_f1:.4f}")
print("Validation accuracy:", accuracy_score(y_valid, valid_preds))
print("Validation precision:", precision_score(y_valid, valid_preds))
print("Validation recall:", recall_score(y_valid, valid_preds))
print("Validation F1:", f1_score(y_valid, valid_preds))
print("Validation ROC-AUC:", roc_auc_score(y_valid, valid_probs))
print()
print(classification_report(y_valid, valid_preds))
print()
print(
    "Confusion matrix:\n",
    confusion_matrix(y_valid, valid_preds)
)

Best threshold from train: 0.83
Train F1 at best threshold: 0.6425
Validation accuracy: 0.6512537488792011
Validation precision: 0.5167227039462042
Validation recall: 0.855874717360355
Validation F1: 0.6443985560932549
Validation ROC-AUC: 0.7454657120871406

              precision    recall  f1-score   support

           0       0.86      0.53      0.66     40804
           1       0.52      0.86      0.64     23882

    accuracy                           0.65     64686
   macro avg       0.69      0.69      0.65     64686
weighted avg       0.74      0.65      0.65     64686


Confusion matrix:
 [[21687 19117]
 [ 3442 20440]]


**Conclusions**
* Sentence-BERT embeddings successfully capture semantic similarity between questions.
* Using cosine similarity alone provides reasonable discrimination between duplicate and non-duplicate question pairs (ROC-AUC ≈ 0.75).
* The model achieves high recall (≈0.86), meaning that most duplicate questions are correctly identified.
* However, the relatively low precision indicates that many non-duplicate pairs are incorrectly classified as duplicates.
* These results suggest that cosine similarity is a strong semantic feature, but it is insufficient as a standalone classifier.
* In the following experiments, the cosine similarity score will be used as an additional feature together with other text-based and engineered features.

In [ ]:
df[["question1", "question2", "is_duplicate", "bert_cosine_similarity"]].to_csv(feature_file, index=False)

print("Saved cosine similarity feature to:")
print(feature_file)

Saved cosine similarity feature to:
/content/drive/MyDrive/ML Course/Homeworks/Quora duplicates/quora_with_bert_cosine_similarity.csv
